In [1]:
%load_ext dotenv
%dotenv /home/aurora/.env
%matplotlib inline

In [2]:
import numpy as np
import pandas as pd
import datetime as dt
from google.cloud import bigquery
import re
import pymongo
import os
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
from flatten_json import flatten
#warnings.filterwarnings('ignore')

In [3]:
# Time
start = dt.datetime(2019,4,5)
end = dt.datetime(2019,5,27)
print(start,end,end-start)

2019-04-05 00:00:00 2019-05-27 00:00:00 52 days, 0:00:00


In [4]:
cursor = pymongo.MongoClient("mongodb://" + os.environ['user'] + ':' + 
                             os.environ['pass'] + '@' + os.environ['db1']  + "/?authSource=" + 
                             os.environ['dbname'])
c_users = cursor.superstars.users
aw_users = []
for documents in c_users.find({'created_at': {'$lt': end, '$gte': start}},{"sign_up_details":1, "created_at":1,"login_details":1}):
    aw_users.append(documents)
dic_flattened = [flatten(d) for d in aw_users]
df_users = pd.DataFrame(dic_flattened)
df_users = df_users[df_users["sign_up_details_app_platform"] == "UNITY_Android"]
users = df_users[["_id","created_at","sign_up_details_device_id","login_details_last_request_at"]]
users.columns = ["user_id","createtime","device_id","last_request"]
users.head()

,user_id,createtime,device_id,last_request
0,5ca6adb3b65b15544e169963,2019-04-05 01:21:55.931,ba643254ecf37ea8a8453a12ab14e7a8,2019-04-05 01:22:16.059
1,5ca6c1bc8c899454486ac253,2019-04-05 02:47:24.829,3ac01ce3f63c2c857064702d99c80541,2019-04-05 03:09:53.215
2,5ca6d01fd468e03534a80f6d,2019-04-05 03:48:47.021,2534dd971bd5601f2985c94d1da74cc9,2019-04-05 03:51:56.000
4,5ca6e3fc800ebb353a469cd1,2019-04-05 05:13:32.630,bdfe19ca2338a8176670c5c79695b36f,2019-04-05 10:11:34.754
5,5ca6e9a3acb9c30617a1e75f,2019-04-05 05:37:39.265,a395477c3bd6830b7e0c68571da67ed8,2019-04-05 05:37:44.384


In [5]:
users.sort_values(['device_id','createtime'],inplace = True)
users = users.drop_duplicates('device_id')

/home/aurora/miniconda3/lib/python3.7/site-packages/ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  """Entry point for launching an IPython kernel.


In [6]:
users_d3 = users[users['last_request']-users['createtime']>'72:00:00']

In [7]:
users_d15 = users[users['last_request']-users['createtime']>'360:00:00']

In [8]:
users_d45 = users[users['last_request']-users['createtime']>'1080:00:00']

In [9]:
team_cursor = cursor.superstars.teams
aw_teams = []
for documents in team_cursor.find({'created_at': {'$lt': end, '$gte': start}},{"user":1, "created_at":1}):
    aw_teams.append(documents)
dic_flattened = [flatten(d) for d in aw_teams]
df_teams = pd.DataFrame(dic_flattened)
teams = df_teams[df_teams["user"].isin(users["user_id"])]
teams = teams[["_id","user","created_at"]]
teams.columns = ["team_id", "user_id", "team_created_at"]
teams.head()

,team_id,user_id,team_created_at
0,5ca6adb4b65b15544e169988,5ca6adb3b65b15544e169963,2019-04-05 01:21:56.036
1,5ca6c1bc8c899454486ac278,5ca6c1bc8c899454486ac253,2019-04-05 02:47:24.922
2,5ca6d01fd468e03534a80f92,5ca6d01fd468e03534a80f6d,2019-04-05 03:48:47.117
4,5ca6e3fc800ebb353a469cf6,5ca6e3fc800ebb353a469cd1,2019-04-05 05:13:32.725
5,5ca6e9a3acb9c30617a1e784,5ca6e9a3acb9c30617a1e75f,2019-04-05 05:37:39.347


In [10]:
users_team_d3 = pd.merge(teams,users_d3,on='user_id')

In [11]:
users_team_d15 = pd.merge(teams,users_d15,on='user_id')

In [12]:
users_team_d45 = pd.merge(teams,users_d45,on='user_id')

In [13]:
con_cursor = cursor.superstars.matches
aw_matches = []
for documents in con_cursor.find({'created_at': {'$lt': end, '$gte': start},"status":3}):
    aw_matches.append(documents)
dic_flattened = [flatten(d) for d in aw_matches]
df_matches = pd.DataFrame(dic_flattened)
df_matches = df_matches.rename(columns={'home_team_id':'team_id'})
matches = df_matches[df_matches["team_id"].isin(teams["team_id"])]
matches = matches.loc[:,["team_id","winner_team_id","type","start_time","away_team_wickets"]]

In [14]:
matches.head()

,team_id,winner_team_id,type,start_time,away_team_wickets
66,5ca6c1bc8c899454486ac278,5ca6c1bc8c899454486ac278,CAMPAIGN,2019-04-05 02:47:51.521,10.0
67,5ca6c1bc8c899454486ac278,5ca6c1bc8c899454486ac278,CAMPAIGN,2019-04-05 02:50:50.635,1.0
69,5ca6c1bc8c899454486ac278,5ca6c1bc8c899454486ac278,CAMPAIGN,2019-04-05 02:54:28.205,2.0
71,5ca6c1bc8c899454486ac278,5ca6c1bc8c899454486ac278,CAMPAIGN,2019-04-05 02:58:22.589,5.0
76,5ca6c1bc8c899454486ac278,5c876f2b5b2cd5677477505a,CAMPAIGN,2019-04-05 03:03:09.132,1.0


In [15]:
users_match_d3 = pd.merge(matches,users_team_d3,on='team_id')

In [16]:
users_match_d15 = pd.merge(matches,users_team_d15,on='team_id')

In [17]:
users_match_d45 = pd.merge(matches,users_team_d45,on='team_id')

In [18]:
users_match_d3 = users_match_d3[users_match_d3['type']!='ENTRY_LEAGUE']

In [19]:
users_match_d15 = users_match_d15[users_match_d15['type']!='ENTRY_LEAGUE']

In [20]:
users_match_d45 = users_match_d45[users_match_d45['type']!='ENTRY_LEAGUE']

In [21]:
users_match_d3.drop(['winner_team_id','type','team_created_at','device_id','last_request'],axis=1,inplace=True)

In [22]:
users_match_d15.drop(['winner_team_id','type','team_created_at','device_id','last_request'],axis=1,inplace=True)

In [23]:
users_match_d45.drop(['winner_team_id','type','team_created_at','device_id','last_request'],axis=1,inplace=True)

In [24]:
users_match_d3 = users_match_d3[users_match_d3['start_time']-users_match_d3['createtime']<'72:00:00']

In [25]:
users_match_d15 = users_match_d15[users_match_d15['start_time']-users_match_d15['createtime']<'360:00:00']

In [26]:
users_match_d45 = users_match_d45[users_match_d45['start_time']-users_match_d45['createtime']<'1080:00:00']

In [27]:
total_wickets_d3 = users_match_d3.groupby('user_id')['away_team_wickets'].sum()
total_wickets_d3 = total_wickets_d3.to_frame().reset_index()
total_wickets_d15 = users_match_d15.groupby('user_id')['away_team_wickets'].sum()
total_wickets_d15 = total_wickets_d15.to_frame().reset_index()
total_wickets_d45 = users_match_d45.groupby('user_id')['away_team_wickets'].sum()
total_wickets_d45 = total_wickets_d45.to_frame().reset_index()


In [28]:
total_wickets_d3.describe()

,away_team_wickets
count,12857.000000
mean,62.195769
std,76.897315
min,0.000000
25%,15.000000
50%,33.000000
75%,79.000000
max,831.000000


In [29]:
total_wickets_d15.describe()

,away_team_wickets
count,3940.000000
mean,141.871827
std,256.986482
min,2.000000
25%,15.000000
50%,47.000000
75%,154.000000
max,3647.000000


In [30]:
total_wickets_d45.describe()

,away_team_wickets
count,378.000000
mean,508.222222
std,778.371232
min,10.000000
25%,48.250000
50%,258.000000
75%,674.750000
max,7935.000000
